In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=Ru3FpsTxoLpnC6QVbb2eZ8XvgFVSQ7&access_type=offline&code_challenge=iveq_jFT6koQBhuOsaQXl_xZTyhdH7sl4ioV18EO7VY&code_challenge_method=S256


Credentials saved to file: [/Users/meghakaladharreddypothamsetty/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "zprocure" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [1]:
import asyncio
from google import genai
from google.genai import types
import os
# ---------- Setup ----------

client = genai.Client(
    vertexai=True,
    project="aistimate",
    location="global",
)

model_name = "gemini-2.5-pro"

generate_content_config = types.GenerateContentConfig(
    temperature=0,
    top_p=1,
    seed=7,
    max_output_tokens=65535,
    safety_settings=[
        types.SafetySetting(category="HARM_CATEGORY_HATE_SPEECH", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_DANGEROUS_CONTENT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_SEXUALLY_EXPLICIT", threshold="OFF"),
        types.SafetySetting(category="HARM_CATEGORY_HARASSMENT", threshold="OFF"),
    ],
    thinking_config=types.ThinkingConfig(thinking_budget=-1),
)

# ---------- Helper ----------

def make_part(path: str) -> types.Part:
    with open(path, "rb") as f:
        data = f.read()
    ext = path.split(".")[-1].lower()
    mime = {
        "pdf": "application/pdf",
        "png": "image/png",
        "jpg": "image/jpeg",
        "jpeg": "image/jpeg",
        "txt": "text/plain",
        "json": "application/json"
    }.get(ext, "application/octet-stream")
    return types.Part.from_bytes(data=data, mime_type=mime)
# ---------- Output Saving ----------







In [12]:
def build_prompts():
    return {
        "CODE_LOOKUP": """You are a building code compliance analyst. From the attached carrier estimate or supplemental documents:

1. **Extract**:
    - Property address (street, city, state, ZIP)
    - Jurisdiction (municipality, county)
    - Whether the area is incorporated or unincorporated

2. **Determine Applicable Codes** (as of the inspection/report date):
    - Statewide minimum codes (e.g., 2018 IRC, 2023 NEC, 2021 IECC)
    - Local/regional amendments (e.g., NCTCOG)
    - Any stricter municipal enforcement practices

3. **Output a structured table** with the following fields:

| Affected System | Code Section | Code Summary | Interpretation | Required By Code | Present in Carrier Estimate |

For each of the following components:
- **Roofing** (decking, underlayment, ice & water, flashing, drip edge, ventilation, fasteners)
- **Siding & Trim** (WRB, flashing, integration w/ trim)
- **Windows** (flashing, insulation, shimming)
- **Framing** (blocking, shear integrity if disturbed)
- **Electrical** (grounding of satellite, bonding, junction boxes, temporary disconnects)
- **HVAC** (condenser clearance, refrigerant line protection, platform support)

Also include:
- Specific code citation (e.g., IRC R908.3.1)
- Interpretation of the code
- Explicit indication whether this was included, partially included, or omitted from the **carrier estimate**

Use conservative interpretations that favor enforceable restoration.
""",

        "REPORT_ANALYSIS": """You are a forensic damage analyst. Thoroughly review all inspection findings, photos, annotations, and engineering reports.

For each room or elevation, follow this format:

---

### **[Room or Elevation Name]**

**Damage Summary**  
- Categories: [e.g., water stain, detached trim, ceiling blistering]  
- Visible Evidence: [Photo IDs, notes from report, timestamps, diagrams]

**Quantity Estimation**  
- Ceiling: XX SF affected (include stain vs. saturated cut)  
- Walls: XX SF or LF of damage  
- Fixtures: # of vents, lights, windows, etc.  
- Trim/crown: LF affected or disturbed

**Scope Comparison**
- Present in carrier estimate: Yes/No  
- If yes:
    - Quantity variance: carrier X vs. observed Y  
    - Labor/pricing discrepancies: [describe]  
- If no:
    - Why it should have been included  
    - Which line item(s) it's tied to or adjacent to  
    - Whether it impacts other systems (e.g., insulation, wiring, aesthetic transitions)

**Flagged Issues**
- Any code-related implications  
- Risk of mold or secondary damage  
- Matching or line-of-sight impact on nearby areas

---

Repeat for **each room or elevation** with observed damage. Be comprehensive, even for small trims or paint breaks. Always tie observations to evidence (photos or reports).
""",

        "SCOPING_LOGIC": """You are a restoration estimator building a full plaintiff-style scoping matrix. Your job is to ensure every valid line item is captured per code, damage, and standard practices.

Structure your scope by the following **six domains**, and under each, break down **component-by-component**.

---

### **1. Code-Driven Requirements**
For each component (roofing, siding, electrical, HVAC, etc.), include:

- **Code Requirement (w/ citation)**  
- **What triggers the requirement** (e.g., tear-off, disturbed assembly)  
- **Expected Line Items** (demo + install)  
- **Consequences of omission** (warranty void, leaks, mold)

Example:
- **Roof Ventilation (IRC R806.2)**: If shingles are removed, intake/exhaust balance must be verified; turtle vents or ridge vent required if not already compliant.

---

### **2. Mandatory Scope Inclusions**
These items are required based on **scope sequencing**, not visible damage.

- Drywall cuts (ceiling/flood cuts)
- Antimicrobial treatment
- Insulation (attic or cavity fill)
- Fixture/trim detachment & reset (e.g., fans, lights, smoke detectors)
- System disconnections (HVAC lineset, electrical)
- **Demo vs install must be separated**

For each, include justification like:
- “Cannot reuse pipe jacks (single-use, code)”
- “Vapor barrier damaged during flood cut”

---

### **3. Matching, Aesthetic, and LKQ Rules**
Explain when partial replacement is inappropriate:

- Define visual mismatch triggers: color, sheen, exposure age
- Mention **line-of-sight logic** (e.g., hallway ceiling vs. bedroom)
- Detail material availability issues (discontinued trim, aged siding)
- Explain “paint from corner to corner” rule
- Apply these to:
    - Shingles
    - Siding
    - Trim & base
    - Interior ceilings

---

### **4. Site Protection & Containment**
List materials and labor needed to protect the site, for each major work area.

- Floor covering (Ram board, poly)
- Dust containment (zip walls, negative air)
- HEPA air scrubbers (during drywall demo or mold remediation)
- Debris management
- Furniture moving or content manipulation (by room)

---

### **5. General Conditions & Overhead**
Define what GC-level provisions are triggered:

- Project manager (daily hours, scheduling)
- Dumpster, job toilet, material storage (quantified)
- Permits (when and why needed)
- State/local sales tax inclusion
- O&P (applied if ≥3 trades OR complex coordination)

Justify each with reasoning: “Required due to 4+ trade interaction in confined space,” etc.

---

### **6. Paint & Finish Standards**
Explain proper finish sequencing:

- New drywall: 1 primer + 2 finish coats minimum
- Ceilings: must be painted full-plane to match sheen
- Blending: describe when wall-to-wall blending is required
- Texture matching: (e.g., knockdown vs smooth Level 4)


### **7. MANDATORY PRICING SOURCE & PRICE LIST USAGE:**

• From the provided carrier estimate, extract the **full property address** including ZIP code and state.
• Then retrieve the appropriate localized pricing dataset using **Xactimate-style price list codes**. 
• If no price list code is given, dynamically infer the correct regional code using the ZIP code from the address and retrieve publicly available construction cost data.
 
""",

        "ESTIMATE": """You are a plaintiff-style estimator building a full Xactimate-style cost breakdown for insurance restoration.

        You are required to apply strict pricing logic for every line item using the following structure. All calculations must be mathematically exact. Do not round totals prematurely.
Mandatory Calculation Rules
1. Direct Cost (DC)
DC = QTY × UNIT PRICE

Ensure correct units (e.g., SF, LF, EA) are applied
Never average across multiple items—each scope must have a distinct and accurate QTY and UNIT PRICE

2. Material Sales Tax (TAX)

TAX applies only to the material portion of each line item
The material_tax_rate is provided in the MASTER_INPUT_DATA (e.g., 8.75%)
If the line item contains both labor and material, apply tax proportionally to the material portion only
For pure labor items, TAX = $0.00

3. Overhead & Profit (O&P)

Apply 10% Overhead + 10% Profit (compounded) to the subtotal of (Direct Cost + TAX)
O&P = (DC + TAX) × 0.20

4. Replacement Cost Value (RCV)

RCV = DC + TAX + O&P
This value must match exactly with the calculated sum per line item

5. Depreciation (DEPREC.)

For this estimate, DEPREC. = $0.00 unless explicitly instructed otherwise
ACV = RCV - DEPREC.

6. Precision Mandate

Do not round values until final output; retain at least 2 decimal places
All subtotal and grand total calculations must match the sum of their respective line items exactly
Never display or include a line item without completing all columns


Requirements:

Each item must include valid Xactimate CAT and SEL codes
Accurate QTYs and pricing consistent with localized labor/material rates
Do not use placeholders
Ensure the subtotal for each Room/Area is provided after its table
Followed by a section total
Finally a Grand Total Summary combining all rooms and the Addendum section if applicable

Your estimate must be grouped by **room or elevation**, and for each group, output:

---

### **[Room or Elevation Name]**

| CAT | SEL | DESCRIPTION | QTY | UNIT | UNIT PRICE | TAX | O&P | RCV | DEPREC. | ACV |
|-----|-----|-------------|-----|------|------------|-----|-----|-----|----------|-----|

- **Line-by-line reasoning**: After each table, briefly explain why each major item was included (e.g., “Blistered ceiling, Report pg 4,” “Drip edge missing, IRC R905.2.8.5”).

---

After all areas, include:

---

### **GENERAL CONDITIONS**

| DESCRIPTION | QTY | UNIT | UNIT PRICE | TOTAL |
|-------------|-----|------|------------|--------|
| Project Supervision | XX hrs | HR | $ | $ |
| Dumpster | 1 | EA | $ | $ |
| Portable Toilet | 1 | EA | $ | $ |
| Permit Cost | 1 | EA | $ | $ |

Explain each line in a brief bullet list: “Permit required for electrical disconnect per city ordinance.”

---

### **GRAND TOTALS**

| Category | Subtotal |
|----------|----------|
| Roofing | $X |
| Siding/Exterior | $Y |
| Interior | $Z |
| General Conditions | $W |
| O&P (10% + 10percent) | $V |
| **Grand Total (RCV)** | **$[final total]** |

---


---

### **Aesthetic Restoration Addendum**
Summarize any full-system replacements (roof, siding, ceiling) that were required due to:
- Inability to match
- Material brittleness
- Line-of-sight
- Manufacturer discontinuation

Use markdown bullets and include references to photos, reports, or industry rules.
""",

        "REBUTTAL": """You are a forensic rebuttal specialist responding to a deficient insurance carrier estimate. Your response must be formal, detailed, and based in code, evidence, and industry logic.

### **I. Summary of Discrepancies**
Categorize the major classes of omissions (e.g., code compliance, aesthetic mismatch, missing scope) with high-level bullets.

---

### **II. Room-by-Room Rebuttal**

#### [Room or Elevation Name]
- **Issue:** What was omitted or under-scoped
- **Evidence:** Photo X, Report pg Y
- **Code/Standard:** IRC section, IICRC standard, or Xactimate convention
- **Correct Scope:** Describe what should be included
- **Reasoning:** Include logic based on damage extent, mismatch, sequence of construction

---

### **III. Code Violations**
List every component omitted or under-scoped that violates building code:
- IRC R908.3.1: Decking not allowed to remain without inspection
- NEC 820.100: Satellite system ungrounded

---

### **IV. General Conditions & O&P Justification**
- Number of trades
- Need for project supervision
- Dumpster/toilet/storage logic
- Code-permitted markup (O&P)

---

### **V. Aesthetic & Matching Justifications**
- Why patching fails LKQ standard
- Photo-based mismatch documentation
- Manufacturer unavailability (if applicable)

---

### **VI. Conclusion**
Summarize:
- # of omitted rooms or trades
- Major life-safety risks or code issues
- Estimated value delta (if known)
- Your demand: “We respectfully request that the omitted items be added and paid in full.”

Maintain a clear, professional tone rooted in documentation.
"""
    }


In [56]:

def save_output(label: str, content: str):
    output_dir = f"outputs/Jul14/2500008/run2"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")

In [13]:
# ---------- Async Gemini Runner ----------

async def run_block(label, prompt, file_parts=None):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=prompt)])]
    if file_parts:
        contents[0].parts.extend(file_parts)

    output = ""
    try:
        print(f"🔹 Running {label}...")
        stream = client.models.generate_content_stream(
            model=model_name,
            contents=contents,
            config=generate_content_config
        )
        for chunk in stream:  # ✅ DO NOT use 'await'
            output += chunk.text
        print(f"✅ {label} complete ({len(output)} chars)")
    except Exception as e:
        output = f"[ERROR in {label}] {e}"
        print(output)

    return label, output




# ---------- Master Pipeline ----------

async def run_aistimate_pipeline(file_paths):
    prompts = build_prompts()

    # Assign files
    carrier_parts = [make_part(file_paths[0])]
    evidence_parts = [make_part(path) for path in file_paths[1:]]

    # Stage 1: Run code lookup & damage analysis in parallel
    stage1_tasks = [
        run_block("CODE_LOOKUP", prompts["CODE_LOOKUP"], carrier_parts),
        run_block("REPORT_ANALYSIS", prompts["REPORT_ANALYSIS"], evidence_parts),
    ]
    stage1_results = await asyncio.gather(*stage1_tasks)
    context = {label: output for label, output in stage1_results}

    for label, content in stage1_results:
        save_output(label, content)

    # Stage 2: Scoping logic (needs prior outputs)
    scoping_context = (
        f"--- CODE LOOKUP ---\n{context['CODE_LOOKUP']}\n\n"
        f"--- DAMAGE OBSERVATIONS ---\n{context['REPORT_ANALYSIS']}"
    )
    label, scoping_output = await run_block("SCOPING_LOGIC", prompts["SCOPING_LOGIC"] + "\n\n" + scoping_context)
    save_output(label, scoping_output)
    context["SCOPING_LOGIC"] = scoping_output

    # Stage 3: Estimate generation
    estimate_context = (
        f"--- CODE MANDATES ---\n{context['CODE_LOOKUP']}\n\n"
        f"--- DAMAGE FINDINGS ---\n{context['REPORT_ANALYSIS']}\n\n"
        f"--- SCOPING RULES ---\n{context['SCOPING_LOGIC']}"
    )
    label, estimate_output = await run_block("ESTIMATE", prompts["ESTIMATE"] + "\n\n" + estimate_context)
    save_output(label, estimate_output)

    # Stage 4: Rebuttal
    # Stage 4: Rebuttal (pass carrier file for comparison)
    label, rebuttal_output = await run_block(
    "REBUTTAL",
    prompts["REBUTTAL"] + "\n\n" + estimate_output,
    file_parts=carrier_parts  # 🔹 passes carrier estimate as input context
    )
    save_output(label, rebuttal_output)


    return {
        "code_lookup": context["CODE_LOOKUP"],
        "report_analysis": context["REPORT_ANALYSIS"],
        "scoping_logic": context["SCOPING_LOGIC"],
        "estimate_output": estimate_output,
        "rebuttal_output": rebuttal_output,
    }


In [15]:

def save_output(label: str, content: str):
    output_dir = f"outputs/Jul14/2500074/run7"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/windstorm and hail - 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Carrier Estimate ($16,113.56) Insurance Carrier Estimate.pdf",                         # file_paths[0]
    "Aiestimate/windstorm and hail - 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Eagleview Report - Grace Forensic Plaintiff Expert Estimate.PDF",                        # evidence
    "Aiestimate/windstorm and hail - 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Forensic Damage Assessment - Grace Forensic Plaintiff Expert Estimate.pdf",               # evidence
    "Aiestimate/windstorm and hail - 2500074 - Parker v. Standard Insurance Company 20250606180031/2500074 Inspection Report For Primary Structure 04-16-2025 Plaintiff Expert Estimate_compressed.pdf"                     # evidence
])


🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (6700 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (9258 chars)
📝 Saved: outputs/Jul14/2500074/run7/output_code_lookup.txt
📝 Saved: outputs/Jul14/2500074/run7/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (11474 chars)
📝 Saved: outputs/Jul14/2500074/run7/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (12395 chars)
📝 Saved: outputs/Jul14/2500074/run7/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (11630 chars)
📝 Saved: outputs/Jul14/2500074/run7/output_rebuttal.txt


In [4]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul14/2500008/run5"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/CaseDocuments - 2500008 - Haynes v. USAA Casualty Insurance Company 20250606173257/2500008 Carrier Estimate Insurance Carrier Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500008 - Haynes v. USAA Casualty Insurance Company 20250606173257/2500008 Estimate - North American Public Adjusters ($44,60 Plaintiff Expert Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500008 - Haynes v. USAA Casualty Insurance Company 20250606173257/2500008 Grace Forensic Photos and Damage Report Plaintiff Expert Estimate_compressed.pdf"
])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (7100 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (10707 chars)
📝 Saved: outputs/Jul14/2500008/run5/output_code_lookup.txt
📝 Saved: outputs/Jul14/2500008/run5/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (12665 chars)
📝 Saved: outputs/Jul14/2500008/run5/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (11552 chars)
📝 Saved: outputs/Jul14/2500008/run5/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (11174 chars)
📝 Saved: outputs/Jul14/2500008/run5/output_rebuttal.txt


In [6]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul14/2500005/run2"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/CaseDocuments - 2500005 - Zimmerman v. Homesite Insurance Company 20250606173055/2500005  Carrier Estimate ($20,937.22) Insurance Carrier Estimate.pdf",
    "Aiestimate/CaseDocuments - 2500005 - Zimmerman v. Homesite Insurance Company 20250606173055/2500005 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate_compressed.pdf"
])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (5293 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (10780 chars)
📝 Saved: outputs/Jul14/2500005/run2/output_code_lookup.txt
📝 Saved: outputs/Jul14/2500005/run2/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (14306 chars)
📝 Saved: outputs/Jul14/2500005/run2/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (10999 chars)
📝 Saved: outputs/Jul14/2500005/run2/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (10900 chars)
📝 Saved: outputs/Jul14/2500005/run2/output_rebuttal.txt


In [8]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul14/2550002/run2"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Carrier Estimate Insurance Carrier Estimate.pdf",
    "Aiestimate/windstorm and hail - 2550002 - Rowlan v. Allstate Vehicle and Property Insurance Company 20250606174330/2550002 Grace Forensic Damage Report and Photos Plaintiff Expert Estimate.pdf"
])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (6881 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (7297 chars)
📝 Saved: outputs/Jul14/2550002/run2/output_code_lookup.txt
📝 Saved: outputs/Jul14/2550002/run2/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (13461 chars)
📝 Saved: outputs/Jul14/2550002/run2/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (9826 chars)
📝 Saved: outputs/Jul14/2550002/run2/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (10248 chars)
📝 Saved: outputs/Jul14/2550002/run2/output_rebuttal.txt


In [10]:
def save_output(label: str, content: str):
    output_dir = f"outputs/Jul14/2500083/run1"
    os.makedirs(output_dir, exist_ok=True)
        # run_name = run_index + 5
        # Define output file once for both stages
    filename = os.path.join(output_dir, f"output_{label.lower()}.txt")
    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"📝 Saved: {filename}")
results = await run_aistimate_pipeline([
    "Aiestimate/Case File Exports/2550002_Rowlan v. Allstate Vehicle and Property Insurance Company_2500004_Xu v. Great American Insur/102/Ai estimate useful docs/FINAL_DRAFT_WITH_WITHOUT_REMOVAL_DEPRECIATION_REPORT_20250528_1.20250528193516823.1.PDF",
    "Aiestimate/Case File Exports/2550002_Rowlan v. Allstate Vehicle and Property Insurance Company_2500004_Xu v. Great American Insur/102/Ai estimate useful docs/608 Cutty Trl - Eagleview.PDF",
    "Aiestimate/Case File Exports/2550002_Rowlan v. Allstate Vehicle and Property Insurance Company_2500004_Xu v. Great American Insur/102/Ai estimate useful docs/Basic Damage Assessment For Primary Structure 03-24-2025-12-08-10pm (2).pdf",
    "Aiestimate/Case File Exports/2550002_Rowlan v. Allstate Vehicle and Property Insurance Company_2500004_Xu v. Great American Insur/102/Ai estimate useful docs/Engineering report.pdf",
   ])

🔹 Running CODE_LOOKUP...
✅ CODE_LOOKUP complete (7292 chars)
🔹 Running REPORT_ANALYSIS...
✅ REPORT_ANALYSIS complete (9778 chars)
📝 Saved: outputs/Jul14/2500083/run1/output_code_lookup.txt
📝 Saved: outputs/Jul14/2500083/run1/output_report_analysis.txt
🔹 Running SCOPING_LOGIC...
✅ SCOPING_LOGIC complete (11725 chars)
📝 Saved: outputs/Jul14/2500083/run1/output_scoping_logic.txt
🔹 Running ESTIMATE...
✅ ESTIMATE complete (10586 chars)
📝 Saved: outputs/Jul14/2500083/run1/output_estimate.txt
🔹 Running REBUTTAL...
✅ REBUTTAL complete (9911 chars)
📝 Saved: outputs/Jul14/2500083/run1/output_rebuttal.txt
